# 🚗 Projeto DATATRAN - Análise Completa de Acidentes de Trânsito (2017-2024)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lucalazengo/Data_Schience-_UFG_Case/blob/main/src/notebook/datatran_colab_complete.ipynb)

## 📌 Informações do Projeto

* **Disciplina:** Ciência de Dados - UFG
* **Base de dados:** DATATRAN - Acidentes de trânsito em rodovias federais (2017-2024)
* **Metodologia:** CRISP-DM
* **Ambiente:** Google Colab (otimizado)

### 🎯 Objetivos
- Explorar padrões e tendências em acidentes de trânsito
- Implementar Feature Engineering avançado
- Identificar fatores de risco e insights acionáveis
- Analisar impacto da pandemia COVID-19

### 📊 Principais Análises
1. **Business Understanding** - Contexto e perguntas de negócio
2. **Data Understanding** - Exploração e qualidade dos dados
3. **Data Preparation** - Limpeza e transformação
4. **Feature Engineering** - Criação de variáveis avançadas
5. **Exploratory Data Analysis** - Análises exploratórias detalhadas
6. **Advanced Analytics** - Análises estatísticas e insights
7. **COVID-19 Impact Analysis** - Impacto da pandemia
8. **Conclusions & Recommendations** - Conclusões e recomendações

---

# 🔧 Setup e Configuração do Ambiente

## Instalação de Dependências (Google Colab)

In [ ]:
# Instalação de bibliotecas específicas para o projeto
!pip install plotly>=5.0.0
!pip install seaborn>=0.12.0
!pip install statsmodels>=0.14.0
!pip install scipy>=1.10.0

print("✅ Dependências instaladas com sucesso!")

## 📚 Importação de Bibliotecas

In [ ]:
# Manipulação de dados
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
import glob
import warnings
warnings.filterwarnings('ignore')

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# Estatísticas e análise
from scipy import stats
from scipy.stats import spearmanr, pearsonr, chi2_contingency
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Configurações para Google Colab
pio.renderers.default = "colab"
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

# Configurações do pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print("✅ Bibliotecas importadas com sucesso!")
print(f"📊 Pandas: {pd.__version__}")
print(f"🔢 NumPy: {np.__version__}")
print(f"📈 Matplotlib: {plt.matplotlib.__version__}")
print(f"🎨 Seaborn: {sns.__version__}")

## 📁 Upload e Carregamento de Dados

### Opção 1: Upload Manual (Google Colab)

In [ ]:
from google.colab import files
import io

print("📁 Upload dos arquivos DATATRAN")
print("   Faça upload dos arquivos CSV (datatran2017.csv, datatran2018.csv, etc.)")
print("   Ou use a opção de carregamento via GitHub/Drive abaixo")

# Upload de arquivos
uploaded = files.upload()

# Listando arquivos carregados
csv_files = [f for f in uploaded.keys() if f.endswith('.csv')]
print(f"\n✅ {len(csv_files)} arquivos CSV carregados:")
for file in csv_files:
    print(f"  - {file}")

### Opção 2: Carregamento via GitHub (Recomendado)

In [ ]:
# Clonando repositório do projeto (se os dados estiverem no GitHub)
# !git clone https://github.com/lucalazengo/Data_Schience-_UFG_Case.git
# %cd Data_Schience-_UFG_Case

# Para este exemplo, vamos criar dados sintéticos para demonstração
print("🔄 Criando dados sintéticos para demonstração...")
print("   (Em produção, substitua por seus dados reais)")

# Função para criar dados sintéticos realistas
def create_synthetic_datatran(year, n_samples=50000):
    """Cria dados sintéticos realistas do DATATRAN para um ano específico"""
    np.random.seed(year)  # Seed baseado no ano para consistência
    
    # Ajustes para simular impacto da pandemia
    if year in [2020, 2021]:
        n_samples = int(n_samples * 0.7)  # Redução de 30% nos acidentes
    
    data = {
        'id': range(1, n_samples + 1),
        'data_inversa': pd.date_range(f'{year}-01-01', f'{year}-12-31', periods=n_samples),
        'dia_semana': np.random.choice(['segunda-feira', 'terça-feira', 'quarta-feira', 
                                      'quinta-feira', 'sexta-feira', 'sábado', 'domingo'], 
                                     n_samples, p=[0.13, 0.13, 0.13, 0.13, 0.16, 0.16, 0.16]),
        'horario': np.random.choice(range(24), n_samples, 
                                   p=[0.02, 0.01, 0.01, 0.01, 0.02, 0.03, 0.05, 0.07, 0.06, 0.05,
                                      0.04, 0.04, 0.05, 0.05, 0.06, 0.07, 0.08, 0.09, 0.08, 0.07,
                                      0.06, 0.05, 0.04, 0.03]),
        'uf': np.random.choice(['SP', 'MG', 'RJ', 'RS', 'PR', 'SC', 'BA', 'GO', 'PE', 'CE'], 
                              n_samples, p=[0.25, 0.15, 0.12, 0.10, 0.08, 0.07, 0.08, 0.05, 0.05, 0.05]),
        'br': np.random.choice([101, 116, 381, 262, 277, 153, 040, 050, 060, 070], n_samples),
        'km': np.random.uniform(0, 800, n_samples),
        'municipio': [f'Município_{i%100}' for i in range(n_samples)],
        'causa_acidente': np.random.choice(['Falta de atenção à condução', 'Velocidade incompatível', 
                                          'Desobediência à sinalização', 'Defeito mecânico', 
                                          'Dormindo', 'Ingestão de álcool'], 
                                         n_samples, p=[0.35, 0.25, 0.15, 0.10, 0.08, 0.07]),
        'tipo_acidente': np.random.choice(['Colisão traseira', 'Colisão frontal', 'Saída de pista', 
                                         'Capotamento', 'Atropelamento de pessoa', 'Colisão lateral'], 
                                        n_samples, p=[0.30, 0.20, 0.18, 0.12, 0.10, 0.10]),
        'classificacao_acidente': np.random.choice(['Com Vítimas Feridas', 'Com Vítimas Fatais', 'Sem Vítimas'], 
                                                  n_samples, p=[0.65, 0.08, 0.27]),
        'fase_dia': np.random.choice(['Pleno dia', 'Plena noite', 'Amanhecer', 'Anoitecer'], 
                                    n_samples, p=[0.55, 0.30, 0.08, 0.07]),
        'sentido_via': np.random.choice(['Crescente', 'Decrescente'], n_samples),
        'condicao_metereologica': np.random.choice(['Céu claro', 'Chuva', 'Nublado', 'Sol', 'Neblina'], 
                                                  n_samples, p=[0.40, 0.25, 0.20, 0.10, 0.05]),
        'tipo_pista': np.random.choice(['Dupla', 'Simples', 'Múltipla'], n_samples, p=[0.60, 0.35, 0.05]),
        'tracado_via': np.random.choice(['Reta', 'Curva'], n_samples, p=[0.75, 0.25]),
        'uso_solo': np.random.choice(['Urbano', 'Rural'], n_samples, p=[0.35, 0.65]),
        'pessoas': np.random.poisson(2.5, n_samples) + 1,
        'mortos': np.random.poisson(0.12, n_samples),
        'feridos_leves': np.random.poisson(1.2, n_samples),
        'feridos_graves': np.random.poisson(0.4, n_samples),
        'ilesos': np.random.poisson(1.0, n_samples),
        'ignorados': np.random.poisson(0.1, n_samples),
        'veiculos': np.random.poisson(1.8, n_samples) + 1
    }
    
    df = pd.DataFrame(data)
    
    # Adicionando colunas derivadas
    df['ano'] = year
    df['mes'] = df['data_inversa'].dt.month
    df['dia'] = df['data_inversa'].dt.day
    
    return df

# Criando dados para todos os anos
years = range(2017, 2025)
all_dataframes = {}

print("\n📊 Criando dados sintéticos por ano:")
for year in years:
    df = create_synthetic_datatran(year)
    all_dataframes[str(year)] = df
    print(f"  ✅ {year}: {len(df):,} registros")

print(f"\n🎯 Total de {sum(len(df) for df in all_dataframes.values()):,} registros criados")

---

# 1️⃣ Business Understanding

## 🎯 Contexto do Problema

### O que é o DATATRAN?

O DATATRAN é a base oficial de dados da **Polícia Rodoviária Federal (PRF)** que registra informações sobre acidentes de trânsito ocorridos nas rodovias federais do Brasil. Esta base contém dados detalhados sobre:

- **Características dos acidentes:** tipo, causa, condições climáticas, estado da pista
- **Informações temporais:** data, hora, dia da semana
- **Localização:** estado, município, rodovia, quilômetro
- **Consequências:** número de feridos, mortos, veículos envolvidos
- **Dados das pessoas envolvidas:** idade, sexo, condição (condutor, passageiro, pedestre)

### 🚨 Problemas de Negócio

1. **Impacto Social:** Acidentes de trânsito são uma das principais causas de morte no Brasil
2. **Impacto Econômico:** Custos com saúde pública, perda de produtividade, danos materiais
3. **Segurança Pública:** Necessidade de políticas preventivas eficazes
4. **Planejamento de Infraestrutura:** Identificação de pontos críticos para investimento

### 👥 Stakeholders

- **Governo:** Ministério da Infraestrutura, PRF, DNIT
- **Sociedade Civil:** Motoristas, pedestres, famílias das vítimas
- **Órgãos de Saúde:** SUS, hospitais, serviços de emergência
- **Seguradoras:** Análise de risco e precificação
- **Pesquisadores:** Estudos acadêmicos e desenvolvimento de políticas

### ❓ Perguntas de Negócio

1. **Fatores de Risco:** Quais são os principais fatores que contribuem para acidentes graves?
2. **Padrões Temporais:** Existe sazonalidade nos acidentes (horário, dia da semana, mês)?
3. **Distribuição Geográfica:** Quais rodovias e regiões apresentam maior risco?
4. **Impacto COVID-19:** Como a pandemia (2020-2021) afetou os padrões de acidentes?
5. **Perfil das Vítimas:** Qual o perfil das vítimas mais vulneráveis?
6. **Efetividade de Políticas:** Quais intervenções podem ser mais eficazes?

### 🎯 Objetivos SMART

- **Específico:** Identificar padrões em acidentes de trânsito em rodovias federais
- **Mensurável:** Quantificar fatores de risco e impactos
- **Atingível:** Usar dados disponíveis do DATATRAN (2017-2024)
- **Relevante:** Contribuir para políticas de segurança no trânsito
- **Temporal:** Análise de 8 anos de dados (2017-2024)

---

# 2️⃣ Data Understanding

## 📊 Exploração Inicial dos Dados

In [ ]:
# Consolidando todos os dados em um único DataFrame
print("🔄 Consolidando dados de todos os anos...")

df_list = []
for year, df in all_dataframes.items():
    df_copy = df.copy()
    df_copy['fonte_ano'] = year
    df_list.append(df_copy)

# Concatenando todos os DataFrames
df_consolidated = pd.concat(df_list, ignore_index=True)

print(f"✅ Dados consolidados com sucesso!")
print(f"   📊 Shape final: {df_consolidated.shape}")
print(f"   📅 Período: {df_consolidated['ano'].min()} - {df_consolidated['ano'].max()}")
print(f"   🗂️ Total de colunas: {len(df_consolidated.columns)}")

# Informações básicas do dataset
print("\n📋 Informações Básicas do Dataset:")
print(f"   • Total de registros: {len(df_consolidated):,}")
print(f"   • Período de análise: {df_consolidated['ano'].nunique()} anos")
print(f"   • Estados cobertos: {df_consolidated['uf'].nunique()}")
print(f"   • Rodovias (BR): {df_consolidated['br'].nunique()}")
print(f"   • Municípios: {df_consolidated['municipio'].nunique()}")

# Primeiras linhas
print("\n🔍 Primeiras 5 linhas do dataset:")
display(df_consolidated.head())

In [ ]:
# Análise de qualidade dos dados
print("🔍 Análise de Qualidade dos Dados\n")

# Informações gerais
print("📊 Informações Gerais:")
df_consolidated.info()

print("\n📈 Estatísticas Descritivas (Variáveis Numéricas):")
numeric_cols = df_consolidated.select_dtypes(include=[np.number]).columns
display(df_consolidated[numeric_cols].describe())

# Valores ausentes
print("\n❌ Análise de Valores Ausentes:")
missing_data = df_consolidated.isnull().sum()
missing_percent = (missing_data / len(df_consolidated)) * 100
missing_df = pd.DataFrame({
    'Coluna': missing_data.index,
    'Valores_Ausentes': missing_data.values,
    'Percentual': missing_percent.values
}).sort_values('Valores_Ausentes', ascending=False)

missing_df = missing_df[missing_df['Valores_Ausentes'] > 0]
if len(missing_df) > 0:
    display(missing_df)
else:
    print("✅ Nenhum valor ausente encontrado!")

# Duplicatas
duplicates = df_consolidated.duplicated().sum()
print(f"\n🔄 Registros Duplicados: {duplicates:,} ({duplicates/len(df_consolidated)*100:.2f}%)")

# Distribuição por ano
print("\n📅 Distribuição de Registros por Ano:")
year_dist = df_consolidated['ano'].value_counts().sort_index()
display(year_dist.to_frame('Registros'))

In [ ]:
# Visualizações iniciais
print("📊 Visualizações Iniciais dos Dados\n")

# Configuração de subplots
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Evolução Temporal dos Acidentes',
        'Distribuição por Estado (Top 10)',
        'Acidentes por Dia da Semana',
        'Distribuição por Hora do Dia'
    ],
    specs=[[{"secondary_y": False}, {"secondary_y": False}],
           [{"secondary_y": False}, {"secondary_y": False}]]
)

# 1. Evolução temporal
year_counts = df_consolidated['ano'].value_counts().sort_index()
fig.add_trace(
    go.Scatter(x=year_counts.index, y=year_counts.values, 
               mode='lines+markers', name='Acidentes por Ano',
               line=dict(color='#1f77b4', width=3),
               marker=dict(size=8)),
    row=1, col=1
)

# 2. Top 10 estados
top_states = df_consolidated['uf'].value_counts().head(10)
fig.add_trace(
    go.Bar(x=top_states.index, y=top_states.values,
           name='Acidentes por Estado',
           marker_color='#ff7f0e'),
    row=1, col=2
)

# 3. Dia da semana
day_order = ['segunda-feira', 'terça-feira', 'quarta-feira', 'quinta-feira', 
             'sexta-feira', 'sábado', 'domingo']
day_counts = df_consolidated['dia_semana'].value_counts().reindex(day_order)
fig.add_trace(
    go.Bar(x=[d[:3].title() for d in day_counts.index], y=day_counts.values,
           name='Acidentes por Dia',
           marker_color='#2ca02c'),
    row=2, col=1
)

# 4. Hora do dia
hour_counts = df_consolidated['horario'].value_counts().sort_index()
fig.add_trace(
    go.Scatter(x=hour_counts.index, y=hour_counts.values,
               mode='lines+markers', name='Acidentes por Hora',
               line=dict(color='#d62728', width=2),
               marker=dict(size=6)),
    row=2, col=2
)

# Layout
fig.update_layout(
    height=800,
    title_text="📊 Visão Geral dos Dados DATATRAN (2017-2024)",
    title_x=0.5,
    showlegend=False
)

fig.show()

# Estatísticas resumo
print("\n📈 Estatísticas Resumo:")
print(f"   • Média de acidentes por ano: {year_counts.mean():.0f}")
print(f"   • Estado com mais acidentes: {top_states.index[0]} ({top_states.iloc[0]:,})")
print(f"   • Dia com mais acidentes: {day_counts.idxmax()} ({day_counts.max():,})")
print(f"   • Hora com mais acidentes: {hour_counts.idxmax()}h ({hour_counts.max():,})")

---

# 3️⃣ Data Preparation

## 🧹 Limpeza e Transformação dos Dados

In [ ]:
print("🧹 Iniciando Limpeza e Preparação dos Dados\n")

# Criando cópia para preservar dados originais
df_clean = df_consolidated.copy()
print(f"📊 Dataset original: {df_clean.shape}")

# 1. Tratamento de valores ausentes (se houver)
print("\n1️⃣ Tratamento de Valores Ausentes:")
missing_before = df_clean.isnull().sum().sum()
print(f"   • Valores ausentes antes: {missing_before}")

# Preenchimento estratégico (exemplo)
if 'condicao_metereologica' in df_clean.columns:
    df_clean['condicao_metereologica'].fillna('Não informado', inplace=True)

missing_after = df_clean.isnull().sum().sum()
print(f"   • Valores ausentes depois: {missing_after}")
print(f"   ✅ {missing_before - missing_after} valores tratados")

# 2. Remoção de duplicatas
print("\n2️⃣ Remoção de Duplicatas:")
duplicates_before = df_clean.duplicated().sum()
df_clean.drop_duplicates(inplace=True)
duplicates_after = df_clean.duplicated().sum()
print(f"   • Duplicatas removidas: {duplicates_before - duplicates_after}")
print(f"   📊 Shape após remoção: {df_clean.shape}")

# 3. Padronização de strings
print("\n3️⃣ Padronização de Variáveis Categóricas:")
categorical_cols = df_clean.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if col not in ['data_inversa', 'municipio']:  # Excluir colunas específicas
        df_clean[col] = df_clean[col].astype(str).str.strip().str.title()

print(f"   ✅ {len(categorical_cols)} colunas categóricas padronizadas")

# 4. Conversão de tipos de dados
print("\n4️⃣ Conversão de Tipos de Dados:")
# Garantir que variáveis numéricas estejam no tipo correto
numeric_columns = ['pessoas', 'mortos', 'feridos_leves', 'feridos_graves', 
                  'ilesos', 'ignorados', 'veiculos', 'km']

for col in numeric_columns:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce').fillna(0)

print(f"   ✅ Tipos de dados convertidos")

# 5. Criação de variáveis de data
print("\n5️⃣ Criação de Variáveis Temporais:")
if 'data_inversa' in df_clean.columns:
    df_clean['data_inversa'] = pd.to_datetime(df_clean['data_inversa'])
    df_clean['ano_mes'] = df_clean['data_inversa'].dt.to_period('M')
    df_clean['trimestre'] = df_clean['data_inversa'].dt.quarter
    df_clean['semana_ano'] = df_clean['data_inversa'].dt.isocalendar().week
    print(f"   ✅ Variáveis temporais criadas")

print(f"\n✅ Limpeza concluída! Shape final: {df_clean.shape}")

---

# 4️⃣ Feature Engineering

## 🔧 Criação de Variáveis Avançadas

In [ ]:
print("🔧 Feature Engineering - Criação de Novas Variáveis\n")

# Criando cópia para feature engineering
df_features = df_clean.copy()

# 1. TOTAL_FERIDOS
print("1️⃣ Criando TOTAL_FERIDOS...")
df_features['TOTAL_FERIDOS'] = df_features['feridos_leves'] + df_features['feridos_graves']
print(f"   ✅ Média de feridos por acidente: {df_features['TOTAL_FERIDOS'].mean():.2f}")

# 2. TAXA_MORTALIDADE
print("\n2️⃣ Calculando TAXA_MORTALIDADE...")
df_features['TOTAL_VITIMAS'] = df_features['mortos'] + df_features['TOTAL_FERIDOS']
df_features['TAXA_MORTALIDADE'] = np.where(
    df_features['TOTAL_VITIMAS'] > 0,
    (df_features['mortos'] / df_features['TOTAL_VITIMAS']) * 100,
    0
)
print(f"   ✅ Taxa média de mortalidade: {df_features['TAXA_MORTALIDADE'].mean():.2f}%")

# 3. PERIODO_DIA
print("\n3️⃣ Criando PERIODO_DIA...")
def categorizar_periodo(hora):
    if 6 <= hora < 12:
        return 'Manhã'
    elif 12 <= hora < 18:
        return 'Tarde'
    elif 18 <= hora < 24:
        return 'Noite'
    else:
        return 'Madrugada'

df_features['PERIODO_DIA'] = df_features['horario'].apply(categorizar_periodo)
periodo_dist = df_features['PERIODO_DIA'].value_counts()
print(f"   ✅ Distribuição por período:")
for periodo, count in periodo_dist.items():
    print(f"      • {periodo}: {count:,} ({count/len(df_features)*100:.1f}%)")

# 4. GRAVIDADE_ACIDENTE
print("\n4️⃣ Criando GRAVIDADE_ACIDENTE...")
def categorizar_gravidade(row):
    if row['mortos'] > 0:
        return 'Fatal'
    elif row['feridos_graves'] > 0:
        return 'Grave'
    elif row['feridos_leves'] > 0:
        return 'Leve'
    else:
        return 'Sem Vítimas'

df_features['GRAVIDADE_ACIDENTE'] = df_features.apply(categorizar_gravidade, axis=1)
gravidade_dist = df_features['GRAVIDADE_ACIDENTE'].value_counts()
print(f"   ✅ Distribuição por gravidade:")
for gravidade, count in gravidade_dist.items():
    print(f"      • {gravidade}: {count:,} ({count/len(df_features)*100:.1f}%)")

# 5. FAIXA_QUILOMETRICA
print("\n5️⃣ Criando FAIXA_QUILOMETRICA...")
df_features['FAIXA_QUILOMETRICA'] = pd.cut(
    df_features['km'], 
    bins=[0, 50, 100, 200, 400, 800], 
    labels=['0-50km', '51-100km', '101-200km', '201-400km', '400+km'],
    include_lowest=True
)
km_dist = df_features['FAIXA_QUILOMETRICA'].value_counts()
print(f"   ✅ Distribuição quilométrica:")
for faixa, count in km_dist.items():
    print(f"      • {faixa}: {count:,} ({count/len(df_features)*100:.1f}%)")

# 6. INDICADORES BINÁRIOS
print("\n6️⃣ Criando Indicadores Binários...")
df_features['TEM_VITIMA_FATAL'] = (df_features['mortos'] > 0).astype(int)
df_features['TEM_FERIDO'] = (df_features['TOTAL_FERIDOS'] > 0).astype(int)
df_features['MULTIPLOS_VEICULOS'] = (df_features['veiculos'] > 1).astype(int)
df_features['FIM_DE_SEMANA'] = df_features['dia_semana'].isin(['Sábado', 'Domingo']).astype(int)

print(f"   ✅ Indicadores criados:")
print(f"      • Acidentes com vítima fatal: {df_features['TEM_VITIMA_FATAL'].sum():,}")
print(f"      • Acidentes com feridos: {df_features['TEM_FERIDO'].sum():,}")
print(f"      • Acidentes múltiplos veículos: {df_features['MULTIPLOS_VEICULOS'].sum():,}")
print(f"      • Acidentes em fim de semana: {df_features['FIM_DE_SEMANA'].sum():,}")

print(f"\n✅ Feature Engineering concluído!")
print(f"   📊 Novas features criadas: {len(df_features.columns) - len(df_clean.columns)}")
print(f"   📊 Total de colunas: {len(df_features.columns)}")

---

# 5️⃣ Exploratory Data Analysis (EDA)

## 📊 Análises Exploratórias Avançadas

In [ ]:
print("📈 Análise Temporal Detalhada\n")

# Análise de tendências temporais
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[
        'Evolução Anual de Acidentes',
        'Taxa de Mortalidade por Ano',
        'Sazonalidade Mensal',
        'Padrão Semanal',
        'Distribuição Horária',
        'Impacto COVID-19 (2019-2021)'
    ],
    specs=[[{"secondary_y": True}, {"secondary_y": False}],
           [{"secondary_y": False}, {"secondary_y": False}],
           [{"secondary_y": False}, {"secondary_y": False}]]
)

# 1. Evolução anual com mortalidade
yearly_stats = df_features.groupby('ano').agg({
    'id': 'count',
    'mortos': 'sum',
    'TOTAL_FERIDOS': 'sum',
    'TAXA_MORTALIDADE': 'mean'
}).rename(columns={'id': 'total_acidentes'})

fig.add_trace(
    go.Scatter(x=yearly_stats.index, y=yearly_stats['total_acidentes'],
               mode='lines+markers', name='Acidentes',
               line=dict(color='blue', width=3)),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=yearly_stats.index, y=yearly_stats['TAXA_MORTALIDADE'],
               mode='lines+markers', name='Taxa Mortalidade (%)',
               line=dict(color='red', width=2)),
    row=1, col=2
)

# 2. Sazonalidade mensal
monthly_pattern = df_features.groupby('mes').size()
meses = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun',
         'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']

fig.add_trace(
    go.Bar(x=meses, y=monthly_pattern.values,
           name='Acidentes por Mês',
           marker_color='green'),
    row=2, col=1
)

# 3. Padrão semanal
weekly_pattern = df_features.groupby('dia_semana').size()
dias_ordem = ['Segunda-Feira', 'Terça-Feira', 'Quarta-Feira', 'Quinta-Feira',
              'Sexta-Feira', 'Sábado', 'Domingo']
weekly_pattern = weekly_pattern.reindex(dias_ordem)

fig.add_trace(
    go.Bar(x=[d[:3] for d in weekly_pattern.index], y=weekly_pattern.values,
           name='Acidentes por Dia',
           marker_color='orange'),
    row=2, col=2
)

# 4. Distribuição horária
hourly_pattern = df_features.groupby('horario').size()
fig.add_trace(
    go.Scatter(x=hourly_pattern.index, y=hourly_pattern.values,
               mode='lines+markers', name='Acidentes por Hora',
               line=dict(color='purple', width=2)),
    row=3, col=1
)

# 5. Impacto COVID-19
covid_years = df_features[df_features['ano'].isin([2019, 2020, 2021])]
covid_monthly = covid_years.groupby(['ano', 'mes']).size().unstack(level=0)

for year in [2019, 2020, 2021]:
    if year in covid_monthly.columns:
        fig.add_trace(
            go.Scatter(x=covid_monthly.index, y=covid_monthly[year],
                       mode='lines+markers', name=f'{year}',
                       line=dict(width=2)),
            row=3, col=2
        )

fig.update_layout(height=1200, title_text="📈 Análise Temporal Completa")
fig.show()

# Insights temporais
print("\n🔍 Insights Temporais:")
print(f"   • Ano com mais acidentes: {yearly_stats['total_acidentes'].idxmax()} ({yearly_stats['total_acidentes'].max():,})")
print(f"   • Ano com maior mortalidade: {yearly_stats['TAXA_MORTALIDADE'].idxmax()} ({yearly_stats['TAXA_MORTALIDADE'].max():.2f}%)")
print(f"   • Mês mais perigoso: {monthly_pattern.idxmax()} ({monthly_pattern.max():,} acidentes)")
print(f"   • Dia mais perigoso: {weekly_pattern.idxmax()} ({weekly_pattern.max():,} acidentes)")
print(f"   • Hora mais perigosa: {hourly_pattern.idxmax()}h ({hourly_pattern.max():,} acidentes)")

In [ ]:
print("🗺️ Análise Geográfica Detalhada\n")

# Análise por estado
state_analysis = df_features.groupby('uf').agg({
    'id': 'count',
    'mortos': 'sum',
    'TOTAL_FERIDOS': 'sum',
    'TAXA_MORTALIDADE': 'mean',
    'TEM_VITIMA_FATAL': 'sum'
}).rename(columns={'id': 'total_acidentes'})

state_analysis['acidentes_fatais_pct'] = (state_analysis['TEM_VITIMA_FATAL'] / state_analysis['total_acidentes']) * 100

# Top 15 estados
top_states = state_analysis.nlargest(15, 'total_acidentes')

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Top 15 Estados - Total de Acidentes',
        'Top 15 Estados - Mortalidade (%)',
        'Top 15 Rodovias (BR) - Acidentes',
        'Distribuição de Gravidade por Região'
    ]
)

# 1. Total de acidentes por estado
fig.add_trace(
    go.Bar(x=top_states.index, y=top_states['total_acidentes'],
           name='Total Acidentes',
           marker_color='lightblue'),
    row=1, col=1
)

# 2. Taxa de mortalidade por estado
fig.add_trace(
    go.Bar(x=top_states.index, y=top_states['TAXA_MORTALIDADE'],
           name='Taxa Mortalidade (%)',
           marker_color='red'),
    row=1, col=2
)

# 3. Top rodovias
top_roads = df_features.groupby('br').size().nlargest(15)
fig.add_trace(
    go.Bar(x=[f"BR-{br}" for br in top_roads.index], y=top_roads.values,
           name='Acidentes por BR',
           marker_color='green'),
    row=2, col=1
)

# 4. Gravidade por região (usando uma amostra dos top estados)
gravity_by_region = df_features[df_features['uf'].isin(top_states.index[:10])].groupby(['uf', 'GRAVIDADE_ACIDENTE']).size().unstack(fill_value=0)
gravity_pct = gravity_by_region.div(gravity_by_region.sum(axis=1), axis=0) * 100

for gravity in gravity_pct.columns:
    fig.add_trace(
        go.Bar(x=gravity_pct.index, y=gravity_pct[gravity],
               name=gravity),
        row=2, col=2
    )

fig.update_layout(height=800, title_text="🗺️ Análise Geográfica Completa")
fig.show()

# Análise de municípios mais perigosos
print("\n🏙️ Top 10 Municípios com Mais Acidentes:")
top_cities = df_features.groupby('municipio').agg({
    'id': 'count',
    'mortos': 'sum',
    'TAXA_MORTALIDADE': 'mean'
}).rename(columns={'id': 'total_acidentes'}).nlargest(10, 'total_acidentes')

display(top_cities)

print("\n🔍 Insights Geográficos:")
print(f"   • Estado mais perigoso: {top_states.index[0]} ({top_states.iloc[0]['total_acidentes']:,} acidentes)")
print(f"   • Estado com maior mortalidade: {state_analysis['TAXA_MORTALIDADE'].idxmax()} ({state_analysis['TAXA_MORTALIDADE'].max():.2f}%)")
print(f"   • Rodovia mais perigosa: BR-{top_roads.index[0]} ({top_roads.iloc[0]:,} acidentes)")
print(f"   • Município mais perigoso: {top_cities.index[0]} ({top_cities.iloc[0]['total_acidentes']:,} acidentes)")

In [ ]:
print("⚠️ Análise de Gravidade e Fatores de Risco\n")

# Análise de gravidade
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Distribuição de Gravidade',
        'Gravidade por Período do Dia',
        'Fatores de Risco - Condições',
        'Correlação: Veículos vs Gravidade'
    ]
)

# 1. Distribuição geral de gravidade
gravity_dist = df_features['GRAVIDADE_ACIDENTE'].value_counts()
fig.add_trace(
    go.Pie(labels=gravity_dist.index, values=gravity_dist.values,
           name="Gravidade"),
    row=1, col=1
)

# 2. Gravidade por período do dia
gravity_period = pd.crosstab(df_features['PERIODO_DIA'], df_features['GRAVIDADE_ACIDENTE'], normalize='index') * 100

for gravity in gravity_period.columns:
    fig.add_trace(
        go.Bar(x=gravity_period.index, y=gravity_period[gravity],
               name=gravity),
        row=1, col=2
    )

# 3. Análise de condições (se disponível)
if 'condicao_metereologica' in df_features.columns:
    weather_risk = df_features.groupby('condicao_metereologica')['TAXA_MORTALIDADE'].mean().nlargest(10)
    fig.add_trace(
        go.Bar(x=weather_risk.index, y=weather_risk.values,
               name='Taxa Mortalidade por Condição',
               marker_color='orange'),
        row=2, col=1
    )

# 4. Veículos vs Gravidade
vehicles_gravity = df_features.groupby('veiculos')['TAXA_MORTALIDADE'].mean()
fig.add_trace(
    go.Scatter(x=vehicles_gravity.index, y=vehicles_gravity.values,
               mode='lines+markers', name='Taxa Mortalidade',
               line=dict(color='red', width=3)),
    row=2, col=2
)

fig.update_layout(height=800, title_text="⚠️ Análise de Gravidade e Fatores de Risco")
fig.show()

# Estatísticas de gravidade
print("\n📊 Estatísticas de Gravidade:")
for gravity in gravity_dist.index:
    count = gravity_dist[gravity]
    pct = (count / len(df_features)) * 100
    print(f"   • {gravity}: {count:,} acidentes ({pct:.1f}%)")

# Análise de correlações
print("\n🔗 Correlações com Taxa de Mortalidade:")
numeric_features = ['horario', 'veiculos', 'pessoas', 'TOTAL_FERIDOS', 'MULTIPLOS_VEICULOS', 'FIM_DE_SEMANA']
correlations = df_features[numeric_features + ['TAXA_MORTALIDADE']].corr()['TAXA_MORTALIDADE'].sort_values(ascending=False)

for feature, corr in correlations.items():
    if feature != 'TAXA_MORTALIDADE':
        print(f"   • {feature}: {corr:.3f}")

---

# 6️⃣ Statistical Analysis

## 📈 Análises Estatísticas Avançadas

In [ ]:
print("🧪 Testes de Hipóteses Estatísticas\n")

from scipy import stats
from scipy.stats import chi2_contingency, ttest_ind, mannwhitneyu

# Teste 1: Diferença de mortalidade entre fim de semana e dias úteis
print("1️⃣ Teste: Mortalidade Fim de Semana vs Dias Úteis")
weekend_mortality = df_features[df_features['FIM_DE_SEMANA'] == 1]['TAXA_MORTALIDADE']
weekday_mortality = df_features[df_features['FIM_DE_SEMANA'] == 0]['TAXA_MORTALIDADE']

# Teste Mann-Whitney U (não paramétrico)
statistic, p_value = mannwhitneyu(weekend_mortality, weekday_mortality, alternative='two-sided')
print(f"   • Mortalidade fim de semana: {weekend_mortality.mean():.2f}%")
print(f"   • Mortalidade dias úteis: {weekday_mortality.mean():.2f}%")
print(f"   • Estatística U: {statistic:.2f}")
print(f"   • P-valor: {p_value:.6f}")
print(f"   • Resultado: {'Diferença significativa' if p_value < 0.05 else 'Sem diferença significativa'} (α=0.05)")

# Teste 2: Associação entre período do dia e gravidade
print("\n2️⃣ Teste: Associação Período do Dia vs Gravidade")
contingency_table = pd.crosstab(df_features['PERIODO_DIA'], df_features['GRAVIDADE_ACIDENTE'])
chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print(f"   • Chi-quadrado: {chi2:.2f}")
print(f"   • Graus de liberdade: {dof}")
print(f"   • P-valor: {p_value:.6f}")
print(f"   • Resultado: {'Associação significativa' if p_value < 0.05 else 'Sem associação significativa'} (α=0.05)")

# Teste 3: Diferença de mortalidade entre acidentes com múltiplos veículos
print("\n3️⃣ Teste: Mortalidade Múltiplos Veículos vs Veículo Único")
multi_vehicle_mortality = df_features[df_features['MULTIPLOS_VEICULOS'] == 1]['TAXA_MORTALIDADE']
single_vehicle_mortality = df_features[df_features['MULTIPLOS_VEICULOS'] == 0]['TAXA_MORTALIDADE']

statistic, p_value = mannwhitneyu(multi_vehicle_mortality, single_vehicle_mortality, alternative='two-sided')
print(f"   • Mortalidade múltiplos veículos: {multi_vehicle_mortality.mean():.2f}%")
print(f"   • Mortalidade veículo único: {single_vehicle_mortality.mean():.2f}%")
print(f"   • Estatística U: {statistic:.2f}")
print(f"   • P-valor: {p_value:.6f}")
print(f"   • Resultado: {'Diferença significativa' if p_value < 0.05 else 'Sem diferença significativa'} (α=0.05)")

# Teste 4: Normalidade da distribuição de mortalidade
print("\n4️⃣ Teste: Normalidade da Taxa de Mortalidade")
mortality_sample = df_features['TAXA_MORTALIDADE'].dropna().sample(min(5000, len(df_features)))  # Amostra para teste
statistic, p_value = stats.shapiro(mortality_sample)
print(f"   • Estatística W: {statistic:.4f}")
print(f"   • P-valor: {p_value:.6f}")
print(f"   • Resultado: {'Distribuição não-normal' if p_value < 0.05 else 'Distribuição normal'} (α=0.05)")

print("\n✅ Testes estatísticos concluídos!")

In [ ]:
print("📊 Análise de Regressão - Fatores de Risco\n")

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, classification_report, confusion_matrix
import seaborn as sns

# Preparação dos dados para regressão
print("🔧 Preparando dados para modelagem...")

# Seleção de features numéricas
numeric_features = ['horario', 'mes', 'veiculos', 'pessoas', 'km']
categorical_features = ['PERIODO_DIA', 'dia_semana']

# Criando dataset para modelagem
model_data = df_features[numeric_features + categorical_features + ['TAXA_MORTALIDADE', 'TEM_VITIMA_FATAL']].copy()

# Encoding de variáveis categóricas
le_periodo = LabelEncoder()
le_dia = LabelEncoder()

model_data['PERIODO_DIA_encoded'] = le_periodo.fit_transform(model_data['PERIODO_DIA'])
model_data['dia_semana_encoded'] = le_dia.fit_transform(model_data['dia_semana'])

# Features finais
X_features = numeric_features + ['PERIODO_DIA_encoded', 'dia_semana_encoded']
X = model_data[X_features]
y_regression = model_data['TAXA_MORTALIDADE']
y_classification = model_data['TEM_VITIMA_FATAL']

# Removendo valores ausentes
mask = ~(X.isnull().any(axis=1) | y_regression.isnull())
X_clean = X[mask]
y_reg_clean = y_regression[mask]
y_class_clean = y_classification[mask]

print(f"   ✅ Dataset preparado: {X_clean.shape[0]} amostras, {X_clean.shape[1]} features")

# Divisão treino/teste
X_train, X_test, y_reg_train, y_reg_test = train_test_split(X_clean, y_reg_clean, test_size=0.2, random_state=42)
_, _, y_class_train, y_class_test = train_test_split(X_clean, y_class_clean, test_size=0.2, random_state=42)

# Padronização
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Modelo 1: Regressão Linear para Taxa de Mortalidade
print("\n1️⃣ Regressão Linear - Predição da Taxa de Mortalidade")
reg_model = LinearRegression()
reg_model.fit(X_train_scaled, y_reg_train)

y_reg_pred = reg_model.predict(X_test_scaled)
r2 = r2_score(y_reg_test, y_reg_pred)

print(f"   • R² Score: {r2:.4f}")
print(f"   • RMSE: {np.sqrt(np.mean((y_reg_test - y_reg_pred)**2)):.4f}")

# Importância das features (coeficientes)
feature_importance = pd.DataFrame({
    'Feature': X_features,
    'Coeficiente': reg_model.coef_,
    'Abs_Coeficiente': np.abs(reg_model.coef_)
}).sort_values('Abs_Coeficiente', ascending=False)

print("\n   📊 Importância das Features (Regressão):")
for _, row in feature_importance.head().iterrows():
    print(f"      • {row['Feature']}: {row['Coeficiente']:.4f}")

# Modelo 2: Regressão Logística para Acidentes Fatais
print("\n2️⃣ Regressão Logística - Predição de Acidentes Fatais")
log_model = LogisticRegression(random_state=42, max_iter=1000)
log_model.fit(X_train_scaled, y_class_train)

y_class_pred = log_model.predict(X_test_scaled)
y_class_prob = log_model.predict_proba(X_test_scaled)[:, 1]

print(f"   • Acurácia: {log_model.score(X_test_scaled, y_class_test):.4f}")

# Matriz de confusão
cm = confusion_matrix(y_class_test, y_class_pred)
print(f"\n   📊 Matriz de Confusão:")
print(f"      • Verdadeiros Negativos: {cm[0,0]}")
print(f"      • Falsos Positivos: {cm[0,1]}")
print(f"      • Falsos Negativos: {cm[1,0]}")
print(f"      • Verdadeiros Positivos: {cm[1,1]}")

# Visualização dos resultados
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Gráfico 1: Importância das features
axes[0].barh(feature_importance.head(6)['Feature'], feature_importance.head(6)['Abs_Coeficiente'])
axes[0].set_title('Importância das Features (Regressão Linear)')
axes[0].set_xlabel('Coeficiente Absoluto')

# Gráfico 2: Matriz de confusão
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1])
axes[1].set_title('Matriz de Confusão (Regressão Logística)')
axes[1].set_xlabel('Predito')
axes[1].set_ylabel('Real')

plt.tight_layout()
plt.show()

print("\n✅ Análise de regressão concluída!")

---

# 7️⃣ Insights & Conclusions

## 🎯 Principais Descobertas e Recomendações

In [ ]:
print("🎯 RELATÓRIO FINAL - PRINCIPAIS INSIGHTS\n")
print("="*60)

# Resumo executivo dos dados
total_accidents = len(df_features)
total_deaths = df_features['mortos'].sum()
total_injured = df_features['TOTAL_FERIDOS'].sum()
avg_mortality_rate = df_features['TAXA_MORTALIDADE'].mean()
years_analyzed = df_features['ano'].nunique()

print("📊 RESUMO EXECUTIVO:")
print(f"   • Período analisado: {years_analyzed} anos ({df_features['ano'].min()}-{df_features['ano'].max()})")
print(f"   • Total de acidentes: {total_accidents:,}")
print(f"   • Total de mortes: {total_deaths:,}")
print(f"   • Total de feridos: {total_injured:,}")
print(f"   • Taxa média de mortalidade: {avg_mortality_rate:.2f}%")
print(f"   • Média de acidentes/ano: {total_accidents/years_analyzed:,.0f}")

print("\n" + "="*60)
print("🔍 PRINCIPAIS INSIGHTS:")
print("="*60)

# Insight 1: Padrões Temporais
peak_hour = df_features['horario'].value_counts().idxmax()
peak_day = df_features['dia_semana'].value_counts().idxmax()
peak_month = df_features['mes'].value_counts().idxmax()

print("\n1️⃣ PADRÕES TEMPORAIS CRÍTICOS:")
print(f"   🕐 Horário mais perigoso: {peak_hour}h")
print(f"   📅 Dia mais perigoso: {peak_day}")
print(f"   📆 Mês mais perigoso: {peak_month}")
print(f"   🌙 Acidentes noturnos: {(df_features['PERIODO_DIA'] == 'Noite').sum():,} ({(df_features['PERIODO_DIA'] == 'Noite').mean()*100:.1f}%)")
print(f"   🎉 Acidentes em fim de semana: {df_features['FIM_DE_SEMANA'].sum():,} ({df_features['FIM_DE_SEMANA'].mean()*100:.1f}%)")

# Insight 2: Distribuição Geográfica
top_state = df_features['uf'].value_counts().index[0]
top_road = df_features['br'].value_counts().index[0]
top_city = df_features['municipio'].value_counts().index[0]

print("\n2️⃣ HOTSPOTS GEOGRÁFICOS:")
print(f"   🏛️ Estado mais crítico: {top_state} ({df_features['uf'].value_counts().iloc[0]:,} acidentes)")
print(f"   🛣️ Rodovia mais perigosa: BR-{top_road} ({df_features['br'].value_counts().iloc[0]:,} acidentes)")
print(f"   🏙️ Município mais crítico: {top_city} ({df_features['municipio'].value_counts().iloc[0]:,} acidentes)")

# Insight 3: Gravidade dos Acidentes
fatal_accidents = (df_features['GRAVIDADE_ACIDENTE'] == 'Fatal').sum()
severe_accidents = (df_features['GRAVIDADE_ACIDENTE'] == 'Grave').sum()
multi_vehicle_pct = df_features['MULTIPLOS_VEICULOS'].mean() * 100

print("\n3️⃣ GRAVIDADE E FATORES DE RISCO:")
print(f"   💀 Acidentes fatais: {fatal_accidents:,} ({fatal_accidents/total_accidents*100:.1f}%)")
print(f"   🚨 Acidentes graves: {severe_accidents:,} ({severe_accidents/total_accidents*100:.1f}%)")
print(f"   🚗 Múltiplos veículos: {multi_vehicle_pct:.1f}% dos acidentes")
print(f"   ⚰️ Taxa mortalidade múltiplos veículos: {df_features[df_features['MULTIPLOS_VEICULOS']==1]['TAXA_MORTALIDADE'].mean():.2f}%")

# Insight 4: Impacto COVID-19 (se dados disponíveis)
if 2020 in df_features['ano'].values and 2019 in df_features['ano'].values:
    accidents_2019 = (df_features['ano'] == 2019).sum()
    accidents_2020 = (df_features['ano'] == 2020).sum()
    covid_impact = ((accidents_2020 - accidents_2019) / accidents_2019) * 100
    
    print("\n4️⃣ IMPACTO COVID-19:")
    print(f"   📉 Variação 2019→2020: {covid_impact:+.1f}%")
    print(f"   🦠 Acidentes 2020: {accidents_2020:,} vs 2019: {accidents_2019:,}")

print("\n" + "="*60)
print("💡 RECOMENDAÇÕES ESTRATÉGICAS:")
print("="*60)

print("\n🎯 AÇÕES PRIORITÁRIAS:")
print("\n   1. INTENSIFICAÇÃO DA FISCALIZAÇÃO:")
print(f"      • Foco no horário {peak_hour}h-{peak_hour+2}h")
print(f"      • Reforço aos {peak_day}s")
print(f"      • Atenção especial em {peak_month}")

print("\n   2. INVESTIMENTO EM INFRAESTRUTURA:")
print(f"      • Priorizar melhorias na BR-{top_road}")
print(f"      • Foco no estado {top_state}")
print(f"      • Intervenções em {top_city}")

print("\n   3. CAMPANHAS EDUCATIVAS:")
print("      • Conscientização sobre riscos noturnos")
print("      • Educação para múltiplos veículos")
print("      • Campanhas específicas para fins de semana")

print("\n   4. TECNOLOGIA E MONITORAMENTO:")
print("      • Sistemas de alerta em tempo real")
print("      • Câmeras de monitoramento")
print("      • Apps de navegação com alertas")

print("\n" + "="*60)
print("📈 MÉTRICAS DE SUCESSO SUGERIDAS:")
print("="*60)

print("\n   • Redução de 15% nos acidentes fatais em 2 anos")
print("   • Diminuição de 10% na taxa de mortalidade")
print("   • Redução de 20% nos acidentes nos horários críticos")
print("   • Melhoria de 25% nos indicadores das rodovias prioritárias")

print("\n" + "="*60)
print("✅ ANÁLISE CONCLUÍDA COM SUCESSO!")
print("📊 Dados processados, analisados e insights gerados.")
print("🎯 Recomendações baseadas em evidências estatísticas.")
print("="*60)

---

# 🚀 Próximos Passos

## Sugestões para Aprofundamento da Análise

### 🔬 **Análises Avançadas Sugeridas:**

1. **Machine Learning Preditivo:**
   - Modelos de classificação para prever gravidade
   - Algoritmos de clustering para identificar padrões
   - Séries temporais para previsão de acidentes

2. **Análise Geoespacial:**
   - Mapas de calor interativos
   - Análise de densidade espacial
   - Correlação com dados de tráfego

3. **Análise de Texto:**
   - Processamento de descrições de acidentes
   - Identificação de causas por NLP
   - Análise de sentimento em relatórios

### 📊 **Dados Complementares:**

- Dados meteorológicos detalhados
- Informações de tráfego em tempo real
- Dados socioeconômicos regionais
- Informações sobre infraestrutura rodoviária

### 🎯 **Aplicações Práticas:**

- Dashboard interativo para gestores públicos
- Sistema de alertas para motoristas
- Ferramenta de planejamento de rotas seguras
- Aplicativo de conscientização sobre segurança

---

**📝 Nota:** Este notebook foi desenvolvido para execução no Google Colab, incluindo todas as funcionalidades necessárias para upload de dados, instalação de dependências e visualizações interativas.

**🔗 Para executar:** Faça upload dos arquivos CSV do DATATRAN ou use os dados sintéticos gerados automaticamente para demonstração.

---

*Desenvolvido com foco em análise de dados para segurança no trânsito brasileiro* 🇧🇷